# Diabetes Readmission — EDA

Goal: understand the data and confirm which columns are genuinely known **at/by discharge time** (the only ones we can use as model features, to avoid leakage). Missing values in this dataset are marked with the literal string `"?"`, so we load with `na_values=["?"]`.

In [1]:
import pandas as pd

df2 = pd.read_csv("../data/raw/diabetic_data.csv", na_values=["?"])
df2.shape

/var/folders/71/ntvd5lc57cg2wzsrjb6lsb1r0000gn/T/ipykernel_22502/2035716834.py:3: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv("../data/raw/diabetic_data.csv", na_values=["?"])


(101766, 50)

In [2]:
df2.dtypes

encounter_id                int64
patient_nbr                 int64
race                          str
gender                        str
age                           str
weight                        str
admission_type_id           int64
discharge_disposition_id    int64
admission_source_id         int64
time_in_hospital            int64
payer_code                    str
medical_specialty             str
num_lab_procedures          int64
num_procedures              int64
num_medications             int64
number_outpatient           int64
number_emergency            int64
number_inpatient            int64
diag_1                        str
diag_2                        str
diag_3                        str
number_diagnoses            int64
max_glu_serum                 str
A1Cresult                     str
metformin                     str
repaglinide                   str
nateglinide                   str
chlorpropamide                str
glimepiride                   str
acetohexamide 

In [3]:
df2.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Missing values per column

In [4]:
df2.isna().sum()

encounter_id                    0
patient_nbr                     0
race                         2273
gender                          0
age                             0
weight                      98569
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                  40256
medical_specialty           49949
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                         21
diag_2                        358
diag_3                       1423
number_diagnoses                0
max_glu_serum               96420
A1Cresult                   84748
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                     0
acetohexamide 

## Target: readmitted class balance

In [5]:
df2['readmitted'].value_counts(normalize = True)

readmitted
NO     0.539119
>30    0.349282
<30    0.111599
Name: proportion, dtype: float64

## discharge_disposition_id — expired/hospice check

IDS_mapping.csv shows codes 11 (Expired), 13/14 (Hospice), 19/20/21 (Expired, various settings) represent patients who cannot be readmitted. Checking how many rows this affects.

In [6]:
expired_codes = [11, 13, 14, 19, 20, 21]
df2['discharge_disposition_id'].isin(expired_codes).sum()

np.int64(2423)

In [7]:
unknown_codes = [18, 25, 26]
df2['discharge_disposition_id'].isin(unknown_codes).sum()

np.int64(4680)

## admission_type_id / admission_source_id — missing-like codes check

In [8]:
missing_like_codes = { "admission_type_id" : [5,6,8],
                                         "admission_source_id" : [9,15,17,20,21] }
for col,codes in missing_like_codes.items():
       print(col)
       print(df2[col].isin(codes).sum())
       print()

admission_type_id
10396

admission_source_id
7067



## Demographics

In [9]:
df2['readmit_30_flag'] = (df2['readmitted'] == '<30').astype(int)

In [10]:
df2.groupby('race')["readmit_30_flag"].agg(['mean','count'])

,mean,count
race,,
AfricanAmerican,0.112181,19210
Asian,0.101404,641
Caucasian,0.112906,76099
Hispanic,0.104075,2037
Other,0.096282,1506


In [11]:
df2.groupby('gender')["readmit_30_flag"].agg(['mean','count'])

,mean,count
gender,,
Female,0.112452,54708
Male,0.110615,47055
Unknown/Invalid,0.000000,3


In [12]:
df2.groupby('age')["readmit_30_flag"].agg(['mean','count'])

,mean,count
age,,
[0-10),0.018634,161
[10-20),0.057887,691
[20-30),0.142426,1657
[30-40),0.112318,3775
[40-50),0.106040,9685
[50-60),0.096662,17256
[60-70),0.111284,22483
[70-80),0.117731,26068
[80-90),0.120835,17197


## Visit-history counts — correlation with readmission

In [13]:
visit_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
              'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
df2[visit_cols].corrwith(df2['readmit_30_flag'])

time_in_hospital      0.044199
num_lab_procedures    0.020364
num_procedures       -0.012227
num_medications       0.038432
number_outpatient     0.018893
number_emergency      0.060747
number_inpatient      0.165147
number_diagnoses      0.049524
dtype: float64

## Medication columns — value distribution

In [14]:
med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
            'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
            'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
            'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
            'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
for col in med_cols:
       print(col)
       print(df2[col].value_counts(normalize = True))
       print()

metformin
metformin
No        0.803589
Steady    0.180276
Up        0.010485
Down      0.005650
Name: proportion, dtype: float64

repaglinide
repaglinide
No        0.984877
Steady    0.013600
Up        0.001081
Down      0.000442
Name: proportion, dtype: float64

nateglinide
nateglinide
No        0.993092
Steady    0.006564
Up        0.000236
Down      0.000108
Name: proportion, dtype: float64

chlorpropamide
chlorpropamide
No        0.999155
Steady    0.000776
Up        0.000059
Down      0.000010
Name: proportion, dtype: float64

glimepiride
glimepiride
No        0.948991
Steady    0.045890
Up        0.003213
Down      0.001906
Name: proportion, dtype: float64

acetohexamide
acetohexamide
No        0.99999
Steady    0.00001
Name: proportion, dtype: float64

glipizide
glipizide
No        0.875341
Steady    0.111589
Up        0.007566
Down      0.005503
Name: proportion, dtype: float64

glyburide
glyburide
No        0.895348
Steady    0.091131
Up        0.007979
Down      0.005542
Name

## insulin / metformin vs readmission rate

In [15]:
for col in ['insulin','metformin']:
        print(col)
        print(df2.groupby(col)['readmit_30_flag'].agg(['mean','count']))
        print()

insulin
             mean  count
insulin                 
Down     0.138975  12218
No       0.100374  47383
Steady   0.111284  30849
Up       0.129905  11316

metformin
               mean  count
metformin                 
Down       0.120000    575
No         0.115165  81778
Steady     0.097133  18346
Up         0.082474   1067



## Checking what "missing" means for max_glu_serum / A1Cresult

Before assuming missing = not tested, checking whether the non-missing values include a 'Normal' category distinct from missing (which would confirm missing really means not tested, not tested-and-fine).

In [16]:
df2['max_glu_serum'].value_counts()

max_glu_serum
Norm    2597
>200    1485
>300    1264
Name: count, dtype: int64

In [17]:
df2['glu_tested'] = df2['max_glu_serum'].notna().astype(int)
df2['a1c_tested'] = df2['A1Cresult'].notna().astype(int)
for col in ['glu_tested','a1c_tested']:
       print(col)
       print(df2.groupby(col)["readmit_30_flag"].agg(['mean','count']))
       print()

glu_tested
                mean  count
glu_tested                 
0           0.110931  96420
1           0.123644   5346

a1c_tested
                mean  count
a1c_tested                 
0           0.114233  84748
1           0.098484  17018



## change / diabetesMed vs readmission rate

In [18]:
for col in ['change', 'diabetesMed']:
    print(col)
    print(df2.groupby(col)['readmit_30_flag'].agg(['mean', 'count']))
    print()

change
            mean  count
change                 
Ch      0.118228  47011
No      0.105908  54755

diabetesMed
                 mean  count
diabetesMed                 
No           0.095971  23403
Yes          0.116267  78363



## Diagnosis codes — cardinality check

In [19]:
df2['diag_1'].nunique()

716

## Cleaning

Applying decisions made during EDA above to produce a cleaned dataframe for modeling.

### Drop expired/hospice rows

In [20]:
df2_clean = df2[~df2['discharge_disposition_id'].isin(expired_codes)]
df2_clean.shape

(99343, 53)

### Consolidate unknown-like codes into 'Unknown'

In [21]:
unknown_like_codes_by_col = {
    'discharge_disposition_id': [18, 25, 26],
    'admission_type_id': [5, 6, 8],
    'admission_source_id': [9, 15, 17, 20, 21],
}

for col,codes in unknown_like_codes_by_col.items():
  df2_clean[col] = df2_clean[col].replace(codes,'Unknown')
df2_clean[['discharge_disposition_id', 'admission_type_id', 'admission_source_id']].head()

,discharge_disposition_id,admission_type_id,admission_source_id
0,Unknown,Unknown,1
1,1,1,7
2,1,1,7
3,1,1,7
4,1,1,7


### Drop weight column

In [22]:
df2_clean = df2_clean.drop(columns = ['weight'])
df2_clean.shape

(99343, 52)

### Fill max_glu_serum / A1Cresult missing values with 'Not_Tested'

In [23]:
df2_clean['max_glu_serum'] = df2_clean['max_glu_serum'].fillna('Not_Tested')
df2_clean['A1Cresult'] = df2_clean['A1Cresult'].fillna('Not_Tested')
df2_clean[['max_glu_serum', 'A1Cresult']].isna().sum()

max_glu_serum    0
A1Cresult        0
dtype: int64

### Drop zero-variance medication columns

In [24]:
df2_clean = df2_clean.drop(columns=['examide','citoglipton'])
df2_clean.shape

(99343, 50)

### Fill race missing values with 'Unknown'

In [25]:
df2_clean['race'] = df2_clean['race'].fillna('Unknown')
df2_clean['race'].isna().sum()

np.int64(0)

### Drop Unknown/Invalid gender rows

In [26]:
df2_clean = df2_clean[df2_clean['gender'] != 'Unknown/Invalid']
df2_clean.shape

(99340, 50)

### Fill medical_specialty / payer_code missing values with 'Unknown'

In [27]:
df2_clean['medical_specialty'] = df2_clean['medical_specialty'].fillna('Unknown')
df2_clean['payer_code'] = df2_clean['payer_code'].fillna('Unknown')
df2_clean[['medical_specialty', 'payer_code']].isna().sum()

medical_specialty    0
payer_code           0
dtype: int64

### Group diagnosis codes (diag_1/2/3) into ICD-9 chapter categories

716 distinct raw codes is too high-cardinality and fine-grained to use directly. This groups each code into one of 9 broad clinical categories based on its numeric range, following the standard grouping used in the original research behind this dataset. V/E-prefixed codes (supplementary classification codes, not numeric diagnoses) and anything unrecognized fall into 'Other'; missing codes become 'Missing'.

In [28]:
def map_icd9_to_category(code):
    if pd.isna(code):
        return 'Missing'
    code = str(code)
    if code.startswith('V') or code.startswith('E'):
        return 'Other'
    try:
        code_num = float(code)
    except ValueError:
        return 'Other'
    if 390 <= code_num <= 459 or code_num == 785:
        return 'Circulatory'
    elif 460 <= code_num <= 519 or code_num == 786:
        return 'Respiratory'
    elif 520 <= code_num <= 579 or code_num == 787:
        return 'Digestive'
    elif 250 <= code_num < 251:
        return 'Diabetes'
    elif 800 <= code_num <= 999:
        return 'Injury'
    elif 710 <= code_num <= 739:
        return 'Musculoskeletal'
    elif 580 <= code_num <= 629 or code_num == 788:
        return 'Genitourinary'
    elif 140 <= code_num <= 239:
        return 'Neoplasms'
    else:
        return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df2_clean[col + '_cat'] = df2_clean[col].apply(map_icd9_to_category)

df2_clean = df2_clean.drop(columns=['diag_1', 'diag_2', 'diag_3'])
df2_clean[['diag_1_cat', 'diag_2_cat', 'diag_3_cat']].apply(lambda c: c.value_counts())

,diag_1_cat,diag_2_cat,diag_3_cat
Circulatory,29680,31157,29599
Diabetes,8661,12705,16979
Digestive,9333,4088,3857
Genitourinary,5002,8147,6436
Injury,6851,2383,1897
Missing,20,356,1419
Musculoskeletal,4935,1761,1898
Neoplasms,3131,2326,1662
Other,17793,26027,28588
Respiratory,13934,10390,7005


### Final sanity check

In [29]:
print(df2_clean.shape)
print(df2_clean.isna().sum().sum())

(99340, 50)
0


### Save cleaned data

In [30]:
df2_clean.to_csv('../data/processed/readmission_clean.csv', index=False)